In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import joblib
df=pd.read_csv(r"voitures_occasion.csv")
df = df[(df['Prix'] > 0) & (df['Km'] < 500000) & (df['Annee'] < 2026)]
df=df.drop_duplicates()
df=df.dropna(subset=['Prix'])
df=df.fillna({'Km' : df['Km'].median() })
df=pd.get_dummies(df, columns=['Marque' , 'Modele']) 
x=df.drop('Prix' ,axis = 1)
y=df['Prix']
y = np.log(df['Prix'])
x_train , x_test , y_train , y_test = train_test_split(x,y ,test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=30,   random_state = 42)
model.fit(x_train,y_train)
print(f"R2 Score: {model.score(x_test, y_test):.2f}") 
joblib.dump(model, 'modele_voiture.pkl')
print("✅ تم حفظ الموديل! قلب فالدوسي ديالك غادي تلقى 2 fichiers .pkl")


R2 Score: 0.66
✅ تم حفظ الموديل! قلب فالدوسي ديالك غادي تلقى 2 fichiers .pkl


In [1]:
%%writefile app.py
import streamlit as st
import joblib, pandas as pd, numpy as np

model = joblib.load('modele_voiture.pkl')
colonnes = list(model.feature_names_in_)
marques = [c.replace('Marque_' , '') for c in colonnes if 'Marque_' in c]
models = [c.replace('Modele_','') for c in colonnes if 'Modele_' in c ]
st.title("prediction de votre voiture")
km = st.number_input("Km",0,500000,20000)
anne = st.number_input("Annee",1990,2026,2021)
marque = st.selectbox("Marque" , marques)
Model = st.selectbox("Model", models)
if st.button("Start"):
    df=pd.DataFrame([[km , anne , marque , Model]] , columns=['Km' , 'Annee' , 'Marque', 'Modele'])
    df=pd.get_dummies(df)
    df=df.reindex(columns=colonnes , fill_value=0)
    prix = np.exp(model.predict(df)[0])
    st.success(f"Prix {prix:,.0f} DH")


Writing app.py
